# 4.2 Creating a RunPod Container

Prior to running this file, a RUNPOD API KEY is needed.
An ssh key is also needed. See the runpod-ssh-how-to.docx file in this same directory for instructions and an example on how to do so.

## Configuration

In [2]:
import os
import time
import runpod
import paramiko  # for SSH
from pathlib import Path

# ───────────────────────────────────────────────
# CONFIGURATION
# ───────────────────────────────────────────────
RUNPOD_API_KEY = os.environ["RUNPOD_API_KEY"]  # set in your environment
runpod.api_key = RUNPOD_API_KEY

In [3]:
import runpod
print(runpod.get_gpus())

[{'id': 'AMD Instinct MI300X OAM', 'displayName': 'MI300X', 'memoryInGb': 192}, {'id': 'NVIDIA A100 80GB PCIe', 'displayName': 'A100 PCIe', 'memoryInGb': 80}, {'id': 'NVIDIA A100-SXM4-80GB', 'displayName': 'A100 SXM', 'memoryInGb': 80}, {'id': 'NVIDIA A40', 'displayName': 'A40', 'memoryInGb': 48}, {'id': 'NVIDIA B200', 'displayName': 'B200', 'memoryInGb': 180}, {'id': 'NVIDIA GeForce RTX 3070', 'displayName': 'RTX 3070', 'memoryInGb': 8}, {'id': 'NVIDIA GeForce RTX 3080', 'displayName': 'RTX 3080', 'memoryInGb': 10}, {'id': 'NVIDIA GeForce RTX 3080 Ti', 'displayName': 'RTX 3080 Ti', 'memoryInGb': 12}, {'id': 'NVIDIA GeForce RTX 3090', 'displayName': 'RTX 3090', 'memoryInGb': 24}, {'id': 'NVIDIA GeForce RTX 3090 Ti', 'displayName': 'RTX 3090 Ti', 'memoryInGb': 24}, {'id': 'NVIDIA GeForce RTX 4070 Ti', 'displayName': 'RTX 4070 Ti', 'memoryInGb': 12}, {'id': 'NVIDIA GeForce RTX 4080', 'displayName': 'RTX 4080', 'memoryInGb': 16}, {'id': 'NVIDIA GeForce RTX 4080 SUPER', 'displayName': 'RTX

In [4]:
# Inteligent GPU type selection based on model size using runpod.get_gpus() and the memoryInGb field
# def select_gpu_type(model_name):
#     gpus = runpod.get_gpus()
#     gpu_map = {gpu['displayName']: gpu['id'] for gpu in gpus if gpu['isAvailable']}
    
#     if "70b" in model_name or "gemma-2-70b" in model_name:
#         return gpu_map.get("NVIDIA A100 80GB") or gpu_map.get("NVIDIA A100 40GB")
#     elif "13b" in model_name or "gemma-2-13b" in model_name:
#         return gpu_map.get("NVIDIA RTX A6000")
#     else:
#         return gpu_map.get("NVIDIA GeForce RTX 4090")

In [3]:
POD_NAME       = "lm-eval-pod-test"
IMAGE_NAME     = "runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04"
# GPU_TYPE       = "NVIDIA GeForce RTX 5090" # only 32gb VRAM
# GPU_TYPE       = "NVIDIA H200"
GPU_TYPE       = "NVIDIA A40"
RESULTS_FILE   = "/workspace/results.json"     # inside pod
LOCAL_RESULTS  = Path("results.json")          # where to store results locally

## Pod Creation

In [7]:
# ───────────────────────────────────────────────
# 1. Create the pod
# ───────────────────────────────────────────────
pod = runpod.create_pod(
    name=POD_NAME,
    image_name=IMAGE_NAME,
    gpu_type_id=GPU_TYPE,
    gpu_count=1,
    container_disk_in_gb=200,
    volume_in_gb=0,
    min_vcpu_count=4,
    min_memory_in_gb=16,
    ports="22/tcp,11434/http",  # expose SSH and Ollama
    env={
        "OLLAMA_HOST": "0.0.0.0",
        "PYTHONUNBUFFERED": "1"
    },
    support_public_ip=True,
    start_ssh=True
)

pod_id = pod["id"]
print(f"Created pod: {pod_id}")

raw_response: {'data': {'podFindAndDeployOnDemand': {'id': 'da2aevsmsa1e23', 'imageName': 'runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04', 'env': ['OLLAMA_HOST=0.0.0.0', 'PYTHONUNBUFFERED=1', 'PUBLIC_KEY=ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIB0U12zQAIppWu15PQqpNJHZd7+uFrYwuqP6CuH03FI2 ryan.a.bell2.civ@us.navy.mil\n'], 'machineId': 'ysysvhe2p7ub', 'machine': {'podHostId': 'da2aevsmsa1e23-64411809'}}}}
Created pod: da2aevsmsa1e23


In [8]:
details = runpod.get_pod(pod_id)
print("DEBUG details:", details)


DEBUG details: {'id': 'da2aevsmsa1e23', 'containerDiskInGb': 200, 'costPerHr': 0.4, 'desiredStatus': 'RUNNING', 'dockerArgs': None, 'dockerId': None, 'env': ['OLLAMA_HOST=0.0.0.0', 'PYTHONUNBUFFERED=1', 'PUBLIC_KEY=ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIB0U12zQAIppWu15PQqpNJHZd7+uFrYwuqP6CuH03FI2 ryan.a.bell2.civ@us.navy.mil\n'], 'gpuCount': 1, 'imageName': 'runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04', 'lastStatusChange': 'Rented by User: Thu Sep 25 2025 04:46:34 GMT+0000 (Coordinated Universal Time)', 'machineId': 'ysysvhe2p7ub', 'memoryInGb': 50, 'name': 'lm-eval-pod-test', 'podType': 'RESERVED', 'port': None, 'ports': '22/tcp,11434/http', 'uptimeSeconds': 0, 'vcpuCount': 9, 'volumeInGb': 0, 'volumeMountPath': '/runpod-volume', 'runtime': {'ports': [{'ip': '100.65.24.9', 'isIpPublic': False, 'privatePort': 11434, 'publicPort': 60831, 'type': 'http'}, {'ip': '100.65.24.9', 'isIpPublic': False, 'privatePort': 19123, 'publicPort': 60832, 'type': 'http'}, {'ip': '69.30.

## Wait for pod to become RUNNING and get SSH endpoint

In [11]:
# ───────────────────────────────────────────────
# 2. Wait for pod to become RUNNING and get SSH endpoint
# ───────────────────────────────────────────────
import time

ssh_host = None
ssh_port = None

print("Waiting for pod to become RUNNING and for SSH endpoint to appear...")
while True:
    details = runpod.get_pod(pod_id)
    status = details.get("desiredStatus")
    print(f"  Current status: {status}")

    # If the pod is running, check for a public SSH port
    if status == "RUNNING":
        runtime = details.get("runtime")
        if runtime and runtime.get("ports"):
            for p in runtime["ports"]:
                if p["type"] == "tcp" and p["privatePort"] == 22 and p["isIpPublic"]:
                    ssh_host = p["ip"]
                    ssh_port = p["publicPort"]
                    break
            if ssh_host:
                break   # Exit the loop once we have the SSH endpoint

    time.sleep(10)

print(f"Pod is RUNNING with SSH ready at {ssh_host}:{ssh_port}")




Waiting for pod to become RUNNING and for SSH endpoint to appear...
  Current status: RUNNING
Pod is RUNNING with SSH ready at 69.30.85.132:22198


## Connect to the pod

In [5]:
# ───────────────────────────────────────────────
# 3A Connect to the pod
# ───────────────────────────────────────────────
import os
import time
import paramiko
import runpod

# Connect automatically with Paramiko
ssh_key_path = os.path.expanduser("~/.ssh/id_ed25519")
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())  # auto-accept host fingerprint
ssh.connect(ssh_host, port=ssh_port, username="root", key_filename=ssh_key_path)
print("Connected to pod.")

# Connect with your key (no passphrase)
ssh_key_path = os.path.expanduser("~/.ssh/id_ed25519")
pkey = paramiko.Ed25519Key.from_private_key_file(ssh_key_path)
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(ssh_host, port=ssh_port, username="root", pkey=pkey)
print("SSH connection established.")

Connected to pod.
SSH connection established.


## Provision the pod

In [ ]:
print("Starting provisioning...")

# --- Install prerequisites and Ollama ---
# base_commands = [
#     "apt-get update && apt-get install -y curl git python3-pip lshw",
#     "curl -fsSL https://ollama.com/install.sh | sh",
#     # start Ollama in background and detach so Paramiko doesn't hang
#     "nohup env OLLAMA_HOST=0.0.0.0 ollama serve > /tmp/ollama.log 2>&1 &",
#     "sleep 10"   # give the server time to start
# ]

base_commands = [
    # "apt-get update && apt-get install -y curl git python3-pip lshw",
    "apt update && apt install lshw -y",             # optional
    "curl -fsSL https://ollama.com/install.sh | sh", # install Ollama binary
    "nohup env OLLAMA_HOST=0.0.0.0 ollama serve > /tmp/ollama.log 2>&1 &",
    "sleep 10",
]

for cmd in base_commands:
    print(f"Running: {cmd}")
    stdin, stdout, stderr = ssh.exec_command(cmd)
    print(stdout.read().decode())
    err = stderr.read().decode()
    if err: print("ERROR:", err)

# --- Pull the model ---
model_name = "llama3.2:1b"
print(f"Pulling model {model_name} ...")
stdin, stdout, stderr = ssh.exec_command(f"ollama pull {model_name}")
print(stdout.read().decode())
err = stderr.read().decode()
if err: print("ERROR:", err)

# --- Install evaluation harness ---
stdin, stdout, stderr = ssh.exec_command(
    "pip install lm_eval lm_eval[api] ollama==0.3.3"
)
print(stdout.read().decode())
err = stderr.read().decode()
if err: print("ERROR:", err)

# --- Check until the model appears in 'ollama list' ---
print(f"Waiting for model '{model_name}' to show up in `ollama list`...")
while True:
    stdin, stdout, stderr = ssh.exec_command("ollama list")
    output = stdout.read().decode()
    if model_name.split(":")[0] in output:
        print("Model is ready.")
        break
    print("  Not ready yet, retrying in 10 seconds...")
    time.sleep(10)

print("Provisioning complete.")


Starting provisioning...
Running: apt update && apt install lshw -y
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Reading package lists...
Building dependency tree...
Reading state information...
107 packages can be upgraded. Run 'apt list --upgradable' to see them.
Reading package lists...
Building dependency tree...
Reading state information...
lshw is already the newest version (02.19.git.2021.06.19.996aaad9c7-2build1).
0 upgraded, 0 newly installed, 0 to remove and 107 not upgraded.

ERROR: 




Running: curl -fsSL https://ollama.com/install.sh | sh

ERROR: >>> Cleaning up old version at /usr/local/lib/ollama
>>> Inst

In [9]:
# Pushing the yaml file to the pod
yaml_content = r'''
task: sysengbench
dataset_path: rabell/SysEngBench
dataset_name: null
output_type: generate_until
training_split: null
validation_split: null
test_split: test
doc_to_text: "Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Your response must consist solely of the single letter corresponding to the best answer, chosen from one of A, B, C or D.
Any response other than a single letter (A, B, C, or D) will be considered invalid.\n
Question: {{question}}\n
A. {{choiceA}}\n
B. {{choiceB}}\n
C. {{choiceC}}\n
D. {{choiceD}}\n
Answer:"
doc_to_target: "{{answer}}"

generation_kwargs:
  until: []  # No early stopping for thinking models
  max_gen_toks: 10000  # High limit for reasoning traces
  temperature: 0.0  # Deterministic for evaluation

filter_list:
  - name: "strict-match"
    filter:
      - function: "regex"
        regex_pattern: "([ABCD])"
      - function: "take_first"
metric_list:
  - metric: exact_match
    aggregation: mean
    higher_is_better: true
    ignore_punctuation: true
    ignore_case: true
metadata:
  version: 1.0
dataset_kwargs:
  trust_remote_code: true'''

remote_file = "/root/sysengbench.yaml"
command = f"cat > {remote_file} <<'EOF'\n{yaml_content}\nEOF"

stdin, stdout, stderr = ssh.exec_command(command)
print(stdout.read().decode(), stderr.read().decode())


## Run the eval on the pod

In [12]:
model = "llama3.2:1b"
base_url = "http://localhost:11434/v1/chat/completions"  # Ensure Ollama server is running on this URL
include_path = "./"
tasks = "sysengbench"
output_dir = "output/sysengbench/"
log_samples = True
batch_size = "auto"
temperature = 0.0
apply_chat_template = True

# Construct the benchmark command dynamically
log_samples_flag = "--log_samples" if log_samples else ""
apply_template_flag = "--apply_chat_template" if apply_chat_template else ""

lm_eval_cmd = f"""
lm_eval \
  --model local-chat-completions \
  --model_args model='{model}',base_url='{base_url}',num_concurrent=1 \
  --include_path {include_path} \
  --tasks {tasks} \
  --output {output_dir} \
  {log_samples_flag} \
  --num_fewshot 0 \
  --batch_size {batch_size} \
  --limit 10 \
  --gen_kwargs temperature={temperature} \
  {apply_template_flag}
"""

stdin, stdout, stderr = ssh.exec_command(lm_eval_cmd)
print(stdout.read().decode())
err = stderr.read().decode()
if err: print("ERROR:", err)


local-chat-completions (model=llama3.2:1b,base_url=http://localhost:11434/v1/chat/completions,num_concurrent=1), gen_kwargs: (temperature=0.0), limit: 10.0, num_fewshot: 0, batch_size: auto
|   Tasks   |Version|   Filter   |n-shot|  Metric   |   |Value|   |Stderr|
|-----------|------:|------------|-----:|-----------|---|----:|---|-----:|
|sysengbench|      1|strict-match|     0|exact_match|↑  |  0.9|±  |   0.1|


ERROR: 2025-09-25:05:30:13 INFO     [__main__:348] Including path: ./
2025-09-25:05:30:16 WARNING  [__main__:369]  --limit SHOULD ONLY BE USED FOR TESTING.REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2025-09-25:05:30:16 INFO     [__main__:446] Selected Tasks: ['sysengbench']
2025-09-25:05:30:16 INFO     [evaluator:202] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-09-25:05:30:16 WARNING  [evaluator:214] generation_kwargs: {'temperature': 0.0} specified through cli, these settings will up

## Recover the output files from evals on the pod

In [ ]:
# confirming files are indeed placed in root.
stdin, stdout, stderr = ssh.exec_command(
    "find /root -type f \\( -name 'results_*.json' -o -name 'samples_*.jsonl' \\) 2>/dev/null"
)
matches = stdout.read().decode().strip().splitlines()
print("Found files:", matches)


Found files: ['/root/output/sysengbench/llama3.2__1b/results_2025-09-25T05-30-23.489092.json', '/root/output/sysengbench/llama3.2__1b/samples_sysengbench_2025-09-25T05-30-23.489092.jsonl']


In [15]:
import os
import stat

def sftp_get_dir(sftp, remote_dir, local_dir):
    """
    Recursively download remote_dir from the pod to local_dir,
    preserving the folder structure.
    """
    os.makedirs(local_dir, exist_ok=True)
    for entry in sftp.listdir_attr(remote_dir):
        remote_path = os.path.join(remote_dir, entry.filename)
        local_path  = os.path.join(local_dir,  entry.filename)
        if stat.S_ISDIR(entry.st_mode):
            sftp_get_dir(sftp, remote_path, local_path)  # recurse into subdir
        else:
            sftp.get(remote_path, local_path)
            print(f"Downloaded {remote_path} -> {local_path}")


In [ ]:
# permissions check
for path in ["/root", "/root/output", "/root/output/sysengbench"]:
    stdin, stdout, stderr = ssh.exec_command(f"ls -ld {path}")
    print(path, stdout.read().decode(), stderr.read().decode())


/root drwx------ 1 root root 143 Sep 25 05:30 /root
 
/root/output drwxr-xr-x 3 root root 33 Sep 25 05:30 /root/output
 
/root/output/sysengbench drwxr-xr-x 3 root root 34 Sep 25 05:30 /root/output/sysengbench
 


In [21]:
remote_dir = "/root/output"
local_dir  = "C:/Users/rabel/Desktop/dissertation-outputs/runpod_results"

import os, stat

def sftp_get_dir(sftp, remote_dir, local_dir):
    """
    Recursively download remote_dir from the pod to local_dir,
    preserving the folder structure.
    """
    try:
        sftp.chdir(remote_dir)
    except IOError:
        print(f"Remote directory not found: {remote_dir}")
        return
    os.makedirs(local_dir, exist_ok=True)

    for entry in sftp.listdir_attr(remote_dir):
        remote_path = os.path.join(remote_dir, entry.filename)
        local_path  = os.path.join(local_dir, entry.filename)
        if stat.S_ISDIR(entry.st_mode):
            sftp_get_dir(sftp, remote_path, local_path)
        else:
            sftp.get(remote_path, local_path)
            print(f"Downloaded {remote_path} -> {local_path}")

# ---- usage ----
sftp = ssh.open_sftp()
sftp_get_dir(sftp, remote_dir, local_dir)
sftp.close()

print(f"All files downloaded to: {local_dir}")


Remote directory not found: /root/output\sysengbench
All files downloaded to: C:/Users/rabel/Desktop/dissertation-outputs/runpod_results


In [23]:
import posixpath  # <-- always uses forward slashes
import os, stat

def sftp_get_dir_fixed(sftp, remote_dir, local_dir, depth=0):
    indent = "  " * depth
    print(f"{indent}Entering remote: {remote_dir}")

    try:
        sftp.chdir(remote_dir)
    except IOError as e:
        print(f"{indent}❌ Cannot access {remote_dir}: {e}")
        return

    os.makedirs(local_dir, exist_ok=True)
    print(f"{indent}Local target: {local_dir}")

    entries = sftp.listdir_attr(remote_dir)
    if not entries:
        print(f"{indent}(empty directory)")
        return

    for entry in entries:
        remote_path = posixpath.join(remote_dir, entry.filename)  # ✅ POSIX join
        local_path  = os.path.join(local_dir, entry.filename)      # local join is fine
        print(f"{indent}- {entry.filename} "
              f"{'DIR' if stat.S_ISDIR(entry.st_mode) else 'FILE'}")

        if stat.S_ISDIR(entry.st_mode):
            sftp_get_dir_fixed(sftp, remote_path, local_path, depth + 1)
        else:
            try:
                sftp.get(remote_path, local_path)
                print(f"{indent}  ✅ Downloaded to {local_path}")
            except Exception as e:
                print(f"{indent}  ❌ Failed to download {remote_path}: {e}")


In [24]:
remote_dir = "/root/output"
local_dir  = r"C:\Users\rabel\Desktop\dissertation-outputs\runpod_results"

sftp = ssh.open_sftp()
sftp_get_dir_fixed(sftp, remote_dir, local_dir)
sftp.close()


Entering remote: /root/output
Local target: C:\Users\rabel\Desktop\dissertation-outputs\runpod_results
- sysengbench DIR
  Entering remote: /root/output/sysengbench
  Local target: C:\Users\rabel\Desktop\dissertation-outputs\runpod_results\sysengbench
  - llama3.2__1b DIR
    Entering remote: /root/output/sysengbench/llama3.2__1b
    Local target: C:\Users\rabel\Desktop\dissertation-outputs\runpod_results\sysengbench\llama3.2__1b
    - results_2025-09-25T05-30-23.489092.json FILE
      ✅ Downloaded to C:\Users\rabel\Desktop\dissertation-outputs\runpod_results\sysengbench\llama3.2__1b\results_2025-09-25T05-30-23.489092.json
    - samples_sysengbench_2025-09-25T05-30-23.489092.jsonl FILE
      ✅ Downloaded to C:\Users\rabel\Desktop\dissertation-outputs\runpod_results\sysengbench\llama3.2__1b\samples_sysengbench_2025-09-25T05-30-23.489092.jsonl


## Terminate pod

In [25]:
# ───────────────────────────────────────────────
# 6. Terminate pod
# ───────────────────────────────────────────────
runpod.terminate_pod(pod_id)
print(f"Pod {pod_id} terminated.")

ssh.close()

Pod da2aevsmsa1e23 terminated.


## recovering/ reconnecting to a pod

In [3]:
import os, time, runpod

runpod.api_key = os.environ["RUNPOD_API_KEY"]

def reconnect_by_name(pod_name):
    """
    Find an existing RUNNING pod by its name and return its id, public IP and public SSH port.
    """
    pods = runpod.get_pods()
    for p in pods:
        if p["name"] == pod_name and p["desiredStatus"] == "RUNNING":
            details = runpod.get_pod(p["id"])
            runtime = details.get("runtime", {})
            if runtime and runtime.get("ports"):
                for port in runtime["ports"]:
                    if port["type"] == "tcp" and port["privatePort"] == 22 and port["isIpPublic"]:
                        return {
                            "id": details["id"],
                            "host": port["ip"],
                            "port": port["publicPort"]
                        }
    raise RuntimeError(f"No running pod named '{pod_name}' found.")

# Example usage:
pod_name = "lm-eval-pod-test"   # <-- the name you used in create_pod
conn = reconnect_by_name(pod_name)
pod_id, ssh_host, ssh_port = conn["id"], conn["host"], conn["port"]
print(f"Reconnected to {pod_name} -> {ssh_host}:{ssh_port}")


Reconnected to lm-eval-pod-test -> 69.30.85.132:22198


# Dashboard Attempt

In [ ]:
import os
import time
import stat
import runpod
import paramiko
import pandas as pd
from datetime import datetime
from pathlib import Path

# ───────────────────────────────────────────────
# 1. USER CONFIGURATION
# ───────────────────────────────────────────────
RUNPOD_API_KEY = os.environ["RUNPOD_API_KEY"]
runpod.api_key = RUNPOD_API_KEY

# List of models to evaluate
model_list = [
    # e.g. "llama3.2:1b",
]

# Define evaluation tasks and their output directories
task_configs = [
    ("sysengbench",   "output/sysengbench/"),
    ("sysengbench-a", "output/sysengbench-a/"),
]

IMAGE_NAME   = "runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04"
GPU_TYPE     = "NVIDIA A40"     # GPU type for all pods
NUM_CONCURRENT = 1              # number of concurrent requests per lm_eval run
LOCAL_RESULTS_BASE = Path("results")
LOCAL_RESULTS_BASE.mkdir(exist_ok=True)

# ───────────────────────────────────────────────
# 2. DASHBOARD SETUP
# ───────────────────────────────────────────────
dashboard_columns = [
    "Model", "Task", "Pod ID", "GPU Type",
    "Start Time", "Stop Time", "Status"
]
dashboard_df = pd.DataFrame(columns=dashboard_columns)

def add_dashboard_entry(model, task, pod_id, gpu_type):
    global dashboard_df
    dashboard_df = pd.concat([
        dashboard_df,
        pd.DataFrame([{
            "Model": model,
            "Task": task,
            "Pod ID": pod_id,
            "GPU Type": gpu_type,
            "Start Time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "Stop Time": None,
            "Status": "to do"
        }])
    ], ignore_index=True)

def update_dashboard_status(pod_id, task, status):
    global dashboard_df
    idx = dashboard_df[(dashboard_df["Pod ID"] == pod_id) &
                       (dashboard_df["Task"] == task)].index
    if not idx.empty:
        if status in ("done", "crashed"):
            dashboard_df.loc[idx, "Stop Time"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        dashboard_df.loc[idx, "Status"] = status

def show_dashboard():
    """Display the live dashboard in a Jupyter notebook."""
    from caas_jupyter_tools import display_dataframe_to_user
    display_dataframe_to_user("Pod Run Dashboard", dashboard_df)
    return dashboard_df

# ───────────────────────────────────────────────
# 3. POD UTILITIES
# ───────────────────────────────────────────────
def wait_for_pod_ready(pod_id):
    ssh_host = ssh_port = None
    print("Waiting for pod to be RUNNING & SSH ready...")
    while True:
        details = runpod.get_pod(pod_id)
        if details.get("desiredStatus") == "RUNNING":
            for p in details.get("runtime", {}).get("ports", []):
                if p["type"] == "tcp" and p["privatePort"] == 22 and p["isIpPublic"]:
                    return p["ip"], p["publicPort"]
        time.sleep(10)

def sftp_get_dir_fixed(sftp, remote_dir, local_dir):
    os.makedirs(local_dir, exist_ok=True)
    for entry in sftp.listdir_attr(remote_dir):
        remote_path = f"{remote_dir}/{entry.filename}"
        local_path  = os.path.join(local_dir, entry.filename)
        if stat.S_ISDIR(entry.st_mode):
            sftp_get_dir_fixed(sftp, remote_path, local_path)
        else:
            sftp.get(remote_path, local_path)

def provision_pod(ssh, model):
    """Install Ollama, pull model, install lm_eval inside the pod."""
    cmds = [
        "apt update && apt install -y lshw",
        "curl -fsSL https://ollama.com/install.sh | sh",
        "nohup env OLLAMA_HOST=0.0.0.0 ollama serve > /tmp/ollama.log 2>&1 &",
        "sleep 10",
        f"ollama pull {model}",
        "pip install lm_eval lm_eval[api] ollama==0.3.3"
    ]
    for c in cmds:
        print("Running:", c)
        stdin, stdout, stderr = ssh.exec_command(c)
        stdout.channel.recv_exit_status()
        err = stderr.read().decode()
        if err: print("ERROR:", err)

# ───────────────────────────────────────────────
# 4. POD RUNNER
# ───────────────────────────────────────────────
def run_model_in_pod(model, run_all_tasks=True, single_task=None):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    pod_name  = f"lm-eval-{model.replace(':','-')}-{timestamp}"
    pod = runpod.create_pod(
        name=pod_name,
        image_name=IMAGE_NAME,
        gpu_type_id=GPU_TYPE,
        gpu_count=1,
        container_disk_in_gb=200,
        min_vcpu_count=4,
        min_memory_in_gb=16,
        ports="22/tcp,11434/http",
        env={"OLLAMA_HOST": "0.0.0.0", "PYTHONUNBUFFERED": "1"},
        support_public_ip=True,
        start_ssh=True,
    )
    pod_id = pod["id"]
    print(f"\nCreated pod {pod_id} for model {model}")

    # Dashboard rows for each task
    for task_name, _ in task_configs:
        if run_all_tasks or single_task == task_name:
            add_dashboard_entry(model, task_name, pod_id, GPU_TYPE)

    try:
        # Wait for pod readiness
        ssh_host, ssh_port = wait_for_pod_ready(pod_id)
        key_path = os.path.expanduser("~/.ssh/id_ed25519")
        ssh = paramiko.SSHClient()
        ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        ssh.connect(ssh_host, port=ssh_port, username="root",
                    pkey=paramiko.Ed25519Key.from_private_key_file(key_path))
        print("SSH connected.")
        for task_name, _ in task_configs:
            if run_all_tasks or single_task == task_name:
                update_dashboard_status(pod_id, task_name, "started")

        # Provision environment and model
        provision_pod(ssh, model)

        # Run evaluations
        for task_name, output_dir in task_configs:
            if not run_all_tasks and single_task != task_name:
                continue
            update_dashboard_status(pod_id, task_name, "in progress")
            lm_eval_cmd = f"""
            lm_eval \
              --model local-chat-completions \
              --model_args model='{model}',base_url='http://localhost:11434/v1/chat/completions',num_concurrent={NUM_CONCURRENT} \
              --include_path ./ \
              --tasks {task_name} \
              --output {output_dir} \
              --log_samples \
              --num_fewshot 0 \
              --batch_size auto \
              --gen_kwargs temperature=0.0 \
              --apply_chat_template
            """
            stdin, stdout, stderr = ssh.exec_command(lm_eval_cmd)
            stdout.channel.recv_exit_status()
            err = stderr.read().decode()
            if err: print("ERROR:", err)
            update_dashboard_status(pod_id, task_name, "done")

        # Download results
        sftp = ssh.open_sftp()
        sftp_get_dir_fixed(sftp, "/root/output",
                           LOCAL_RESULTS_BASE / model.replace(":", "_"))
        sftp.close()

    except Exception as e:
        print(f"ERROR while running model {model}: {e}")
        for task_name, _ in task_configs:
            if run_all_tasks or single_task == task_name:
                update_dashboard_status(pod_id, task_name, "crashed")

    finally:
        runpod.terminate_pod(pod_id)
        print(f"Pod {pod_id} terminated.\n")
        ssh.close()

# ───────────────────────────────────────────────
# 5. MAIN LOOP
# ───────────────────────────────────────────────
for model in model_list:
    run_model_in_pod(model, run_all_tasks=True)
    # Or to run a single task:
    # run_model_in_pod(model, run_all_tasks=False, single_task="sysengbench")

# ───────────────────────────────────────────────
# 6. View Dashboard
# ───────────────────────────────────────────────
# In a Jupyter notebook cell, run:
# show_dashboard()


# Checking for files to be present / Taking Inventory

In [22]:
import json
import pandas as pd
from pathlib import Path

def generate_model_task_matrix_with_scores_and_samples(base_output_dir: str) -> pd.DataFrame:
    """
    Build a matrix with:
      * 'Model Folder Name' (safe for filesystem)
      * 'Model Ollama Name' (with ':' restored)
      * One column per task showing ✅/❌ with counts, best score, and sample count.

    Directory structure expected:
        base_output_dir/task_name/model_folder_name/
            results_<timestamp>.json
            samples_<task>_<timestamp>.jsonl
    """
    base_path = Path(base_output_dir)
    if not base_path.exists():
        raise FileNotFoundError(f"Directory does not exist: {base_output_dir}")

    # Discover all tasks and all folder names
    task_names = [p.name for p in base_path.iterdir() if p.is_dir()]
    model_folder_names = set()
    for task in task_names:
        for model_dir in (base_path / task).iterdir():
            if model_dir.is_dir():
                model_folder_names.add(model_dir.name)

    # Prepare matrix with two leading columns
    matrix = pd.DataFrame(index=sorted(model_folder_names),
                          columns=["Model Folder Name", "Model Ollama Name"] + sorted(task_names))

    for model_folder in sorted(model_folder_names):
        # Fill in the two identifying columns
        matrix.at[model_folder, "Model Folder Name"] = model_folder
        # Convert "__" back to ":" for correct Ollama name
        matrix.at[model_folder, "Model Ollama Name"] = model_folder.replace("__", ":")

        for task in task_names:
            model_dir = base_path / task / model_folder
            if not model_dir.exists():
                matrix.at[model_folder, task] = "❌ (0 – max:0.0 – samples:0)"
                continue

            result_files = [f for f in model_dir.iterdir()
                            if f.name.startswith("results_") and f.suffix == ".json"]

            if not result_files:
                matrix.at[model_folder, task] = "❌ (0 – max:0.0 – samples:0)"
                continue

            best_score = 0.0
            best_result_file = None

            # Find highest scoring results file
            for rf in result_files:
                try:
                    with open(rf, "r") as fh:
                        data = json.load(fh)
                    if "results" in data and task in data["results"]:
                        task_data = data["results"][task]
                        score = (task_data.get("exact_match,strict-match")
                                 or task_data.get("exact_match")
                                 or max((v for v in task_data.values() if isinstance(v,(int,float))), default=0.0))
                        if score and score > best_score:
                            best_score = float(score)
                            best_result_file = rf
                except Exception as e:
                    print(f"Warning: could not parse {rf}: {e}")

            n_results = len(result_files)
            samples_exist = False
            max_samples_count = 0

            if best_result_file:
                ts = best_result_file.stem.replace("results_", "")
                expected_samples_prefix = f"samples_{task}_{ts}"
                for f in model_dir.iterdir():
                    if f.name.startswith(expected_samples_prefix) and f.suffix == ".jsonl":
                        samples_exist = True
                        with open(f, "r", encoding="utf-8") as sf:
                            count = sum(1 for _ in sf)
                        max_samples_count = count
                        break

            if best_result_file and samples_exist:
                matrix.at[model_folder, task] = f"✅ ({n_results}) – max:{best_score:.3f} – samples:{max_samples_count}"
            else:
                matrix.at[model_folder, task] = f"❌ ({n_results}) – max:{best_score:.3f} – samples:{max_samples_count}"

    matrix.index.name = "Model Folder Name (index)"
    return matrix


In [ ]:

base_output_dir = Path("downloaded_output")
print(f"Base output dir: {base_output_dir}")

# Generate and prepare matrix
print("Generating matrix...")
matrix_df = generate_model_task_matrix_with_scores_and_samples(str(base_output_dir))

matrix_df = matrix_df.reset_index(drop=True)

pd.set_option('display.expand_frame_repr', False)  # don’t wrap whole rows
pd.set_option('display.max_colwidth', None)        # don’t truncate/wrap cells

# View in Jupyter
display(matrix_df)

# Optional export
# matrix_df.to_csv("model_task_matrix_with_scores_and_samples.csv")


Base output dir: downloaded_output
Generating matrix...


,Model Folder Name,Model Ollama Name,sysengbench,sysengbench-a,sysengbench-b,sysengbench-c,sysengbench-d,sysengbench-osq
0,deepseek-r1__14b,deepseek-r1:14b,✅ (1) – max:0.142 – samples:1144,✅ (2) – max:0.647 – samples:1144,✅ (2) – max:0.092 – samples:1144,✅ (2) – max:0.085 – samples:1144,✅ (2) – max:0.180 – samples:1144,❌ (1) – max:0.000 – samples:0
1,deepseek-r1__32b,deepseek-r1:32b,✅ (1) – max:0.145 – samples:1144,✅ (2) – max:0.656 – samples:1144,✅ (2) – max:0.087 – samples:1144,✅ (2) – max:0.079 – samples:1144,✅ (2) – max:0.176 – samples:1144,❌ (1) – max:0.000 – samples:0
2,deepseek-r1__8b,deepseek-r1:8b,✅ (1) – max:0.222 – samples:1144,✅ (2) – max:0.556 – samples:1144,✅ (2) – max:0.251 – samples:1144,✅ (2) – max:0.051 – samples:1144,✅ (1) – max:0.906 – samples:1144,❌ (1) – max:0.000 – samples:0
3,devstral__24b,devstral:24b,✅ (1) – max:0.931 – samples:1144,✅ (1) – max:0.898 – samples:1144,✅ (1) – max:0.941 – samples:1144,✅ (1) – max:0.941 – samples:1144,✅ (1) – max:0.911 – samples:1144,✅ (1) – max:0.024 – samples:845
4,gemma3__12b,gemma3:12b,✅ (2) – max:0.892 – samples:1144,✅ (2) – max:0.865 – samples:1144,✅ (2) – max:0.894 – samples:1144,✅ (2) – max:0.906 – samples:1144,✅ (1) – max:0.892 – samples:1144,✅ (1) – max:0.015 – samples:845
5,gemma3__1b,gemma3:1b,✅ (2) – max:0.691 – samples:1144,✅ (2) – max:0.748 – samples:1144,✅ (2) – max:0.552 – samples:1144,✅ (2) – max:0.794 – samples:1144,✅ (1) – max:0.753 – samples:1144,❌ (1) – max:0.000 – samples:0
6,gemma3__270m,gemma3:270m,✅ (2) – max:0.136 – samples:1144,✅ (2) – max:0.108 – samples:1144,✅ (2) – max:0.007 – samples:1144,✅ (2) – max:0.317 – samples:1144,✅ (1) – max:0.010 – samples:1144,❌ (1) – max:0.000 – samples:0
7,gemma3__27b,gemma3:27b,✅ (2) – max:0.920 – samples:1144,✅ (2) – max:0.903 – samples:1144,✅ (2) – max:0.918 – samples:1144,✅ (2) – max:0.932 – samples:1144,✅ (1) – max:0.911 – samples:1144,❌ (1) – max:0.000 – samples:0
8,gemma3__4b,gemma3:4b,✅ (2) – max:0.863 – samples:1144,✅ (2) – max:0.828 – samples:1144,✅ (2) – max:0.844 – samples:1144,✅ (2) – max:0.876 – samples:1144,✅ (1) – max:0.847 – samples:1144,✅ (1) – max:0.007 – samples:845
9,gemma3n__e2b,gemma3n:e2b,✅ (2) – max:0.865 – samples:1144,✅ (2) – max:0.816 – samples:1144,✅ (2) – max:0.898 – samples:1144,✅ (2) – max:0.876 – samples:1144,✅ (1) – max:0.792 – samples:1144,❌ (1) – max:0.000 – samples:0


In [27]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime

def summarize_eval_results(base_output_dir: str) -> pd.DataFrame:
    """
    Summarize all results_<timestamp>.json files across models and tasks.

    Extracts:
        - Task, Model Folder, Ollama Name
        - Metric type(s)
        - Best score (preferring strict-match)
        - n-samples
        - Temperature, Max Tokens
        - GPU count
        - Evaluation date

    Returns a tidy DataFrame with one row per results file.
    """
    base_path = Path(base_output_dir)
    if not base_path.exists():
        raise FileNotFoundError(f"Directory not found: {base_output_dir}")

    rows = []
    for results_file in base_path.rglob("results_*.json"):
        try:
            with open(results_file, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"⚠️ Could not parse {results_file}: {e}")
            continue

        # Identify model and task
        task_name = next(iter(data.get("results", {}).keys()), "unknown")
        model_name = data.get("model_name") or data.get("config", {}).get("model_args", "unknown")
        folder_model_name = results_file.parent.name
        ollama_name = folder_model_name.replace("__", ":")

        # Extract metrics
        task_metrics = data.get("results", {}).get(task_name, {})
        metric_keys = list(task_metrics.keys())
        score = None
        metric_type = None

        # Prefer strict-match, then none, then fallback
        for key in ["exact_match,strict-match", "exact_match,none", "exact_match"]:
            if key in task_metrics:
                score = float(task_metrics[key])
                metric_type = key
                break
        if score is None:
            # Try to auto-detect numeric key
            numeric_values = [v for v in task_metrics.values() if isinstance(v, (int, float))]
            score = float(numeric_values[0]) if numeric_values else 0.0
            metric_type = "unknown"

        # Extract n-samples and metadata
        nsamples = data.get("n-samples", {}).get(task_name, {}).get("effective", 0)
        temp = data.get("config", {}).get("gen_kwargs", {}).get("temperature", None)
        max_toks = (
            data.get("configs", {})
            .get(task_name, {})
            .get("generation_kwargs", {})
            .get("max_tokens")
            or data.get("configs", {})
            .get(task_name, {})
            .get("generation_kwargs", {})
            .get("max_gen_toks")
        )
        gpu_info = data.get("pretty_env_info", "")
        gpu_count = gpu_info.count("GPU ")
        date_epoch = data.get("date", 0)
        date_str = datetime.utcfromtimestamp(date_epoch).strftime("%Y-%m-%d %H:%M:%S")

        rows.append({
            "Task": task_name,
            "Model Folder": folder_model_name,
            "Ollama Name": ollama_name,
            "Metric Type": metric_type,
            "Score": round(score, 6),
            "n-Samples": nsamples,
            "Temperature": temp,
            "Max Tokens": max_toks,
            "GPU Count": gpu_count,
            "Eval Date": date_str,
            "Result File": results_file.name,
            "Result Path": str(results_file)
        })

    df = pd.DataFrame(rows)

    # Sort by task → model → date
    df.sort_values(["Task", "Model Folder", "Eval Date"], inplace=True)

    # Optional: choose best (strict-match preferred)
    best_df = (
        df.sort_values(["Metric Type"], key=lambda s: s.apply(lambda x: {"exact_match,strict-match": 0, "exact_match,none": 1, "unknown": 2}.get(x, 3)))
          .groupby(["Task", "Model Folder"], as_index=False)
          .first()
    )

    return best_df


if __name__ == "__main__":
    # Example usage:
    base_output_dir = "../phase4_inference/downloaded_output"
    df = summarize_eval_results(base_output_dir)
    print(df.to_string(index=False))
    # Optionally save
    # df.to_csv("evaluation_summary.csv", index=False)


C:\Users\rabel\AppData\Local\Temp\ipykernel_119704\2902160701.py:74: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  date_str = datetime.utcfromtimestamp(date_epoch).strftime("%Y-%m-%d %H:%M:%S")


           Task          Model Folder          Ollama Name              Metric Type    Score  n-Samples  Temperature  Max Tokens  GPU Count           Eval Date                             Result File                                                                                                         Result Path
    sysengbench           gemma3__12b           gemma3:12b exact_match,strict-match 0.892483       1144          0.0       10000          3 2025-09-25 03:46:00 results_2025-09-25T03-53-48.695567.json               ..\phase4_inference\downloaded_output\sysengbench\gemma3__12b\results_2025-09-25T03-53-48.695567.json
    sysengbench            gemma3__1b            gemma3:1b exact_match,strict-match 0.690559       1144          0.0       10000          3 2025-09-25 03:30:39 results_2025-09-25T03-37-36.067377.json                ..\phase4_inference\downloaded_output\sysengbench\gemma3__1b\results_2025-09-25T03-37-36.067377.json
    sysengbench          gemma3__270m          gemma

### Option 1: Extracting the missing from this one (Any Xs)

In [7]:
def extract_missing_from_matrix(matrix_df: pd.DataFrame) -> dict:
    """
    From the new matrix (with Model Folder Name and Model Ollama Name columns),
    return {ollama_model_name: [missing tasks]}.

    Example:
        {
            "deepseek-r1:14b": ["sysengbench", "sysengbench-a"],
            "gemma3:27b": ["sysengbench-d"]
        }
    """
    missing = {}

    # Identify which columns are actual tasks
    # (everything except our name columns)
    task_columns = [c for c in matrix_df.columns
                    if c not in ("Model Folder Name", "Model Ollama Name")]

    for _, row in matrix_df.iterrows():
        ollama_name = row["Model Ollama Name"]
        missing_tasks = [
            task for task in task_columns
            if isinstance(row[task], str) and row[task].startswith("❌")
        ]
        if missing_tasks:
            missing[ollama_name] = missing_tasks

    return missing


In [8]:
# assuming you already created the matrix with scores and samples
# matrix_df = generate_model_task_matrix_with_scores_and_samples(base_output_dir)

missing_dict = extract_missing_from_matrix(matrix_df)

print(missing_dict)

# Count all model-task pairs
total_permutations = sum(len(tasks) for tasks in missing_dict.values())

print(f"Total model-task permutations: {total_permutations}")

{'deepseek-r1:14b': ['sysengbench-osq'], 'deepseek-r1:32b': ['sysengbench-osq'], 'deepseek-r1:8b': ['sysengbench-d', 'sysengbench-osq'], 'gemma3:1b': ['sysengbench-osq'], 'gemma3:270m': ['sysengbench-osq'], 'gemma3:27b': ['sysengbench-osq'], 'gemma3n:e2b': ['sysengbench-osq'], 'gemma3n:e4b': ['sysengbench-osq'], 'llama3.2:1b': ['sysengbench-osq'], 'llama3.2:3b': ['sysengbench-osq'], 'magistral:24b': ['sysengbench-c', 'sysengbench-osq'], 'mistral:7b': ['sysengbench-osq'], 'mistral:instruct': ['sysengbench-osq'], 'phi4-reasoning:14b': ['sysengbench-osq'], 'phi4-reasoning:plus': ['sysengbench', 'sysengbench-d', 'sysengbench-osq'], 'qwen3:0.6b': ['sysengbench-osq'], 'qwen3:14b': ['sysengbench-osq'], 'qwen3:32b': ['sysengbench-a', 'sysengbench-osq'], 'qwen3:4b': ['sysengbench-osq'], 'qwen3:8b': ['sysengbench-osq']}
Total model-task permutations: 25


### Option 2: Extracting the missing with exemptions (e.g. osq...)

In [9]:
import re
import pandas as pd

def extract_missing_from_matrix(matrix_df: pd.DataFrame) -> dict:
    """
    Return {ollama_model_name: [missing tasks]}, skipping known exemptions.

    Supports:
      1. Global model exemptions (skip entire model).
      2. Task-specific sample count exemptions.
      3. Task-specific model exemptions.
      4. Task-specific pattern exemptions.
    """

    # Exempt a model from ALL benchmark tasks
    GLOBAL_MODEL_EXEMPTIONS = [
        "magistral:24b",     # example
        "phi4-reasoning:plus",   # add more here
        "deepseek-r1:8b",
    ]

    EXEMPTIONS = {
        "sysengbench-osq": {
            "samples": [845],
            "patterns": [r"\(1\)"]
        },

        "sysengbench-c": {
            "models": ["magistral:24b"]
        },
    }

    missing = {}
    task_columns = [
        c for c in matrix_df.columns
        if c not in ("Model Folder Name", "Model Ollama Name")
    ]

    for _, row in matrix_df.iterrows():
        ollama_name = row["Model Ollama Name"]

        # 🔥 GLOBAL MODEL EXEMPTION — skip entire model
        if ollama_name in GLOBAL_MODEL_EXEMPTIONS:
            continue

        missing_tasks = []

        for task in task_columns:
            val = row[task]
            if not isinstance(val, str):
                continue

            exempted = False

            # Task-specific exemptions
            if task in EXEMPTIONS:
                rule = EXEMPTIONS[task]

                # Sample count-based exemption
                if "samples" in rule:
                    for exempt_count in rule["samples"]:
                        if re.search(rf"samples:\s*0*{exempt_count}\b", val):
                            exempted = True
                            break

                # Model-specific exemption at task level
                if not exempted and "models" in rule:
                    if ollama_name in rule["models"]:
                        exempted = True

                # Pattern-based exemption
                if not exempted and "patterns" in rule:
                    for pattern in rule["patterns"]:
                        if re.search(pattern, val):
                            exempted = True
                            break

            # The normal ❌ rule
            if not exempted and val.startswith("❌"):
                missing_tasks.append(task)

        if missing_tasks:
            missing[ollama_name] = missing_tasks

    return missing


In [10]:
missing_dict = extract_missing_from_matrix(matrix_df)
print(missing_dict.get("magistral:24b", []))


[]


In [11]:
# assuming you already created the matrix with scores and samples
# matrix_df = generate_model_task_matrix_with_scores_and_samples(base_output_dir)

missing_dict = extract_missing_from_matrix(matrix_df)

print(missing_dict)

# Count all model-task pairs
total_permutations = sum(len(tasks) for tasks in missing_dict.values())

print(f"Total model-task permutations: {total_permutations}")

{'phi4-reasoning:14b': ['sysengbench-osq'], 'qwen3:0.6b': ['sysengbench-osq'], 'qwen3:14b': ['sysengbench-osq'], 'qwen3:32b': ['sysengbench-a', 'sysengbench-osq'], 'qwen3:4b': ['sysengbench-osq'], 'qwen3:8b': ['sysengbench-osq']}
Total model-task permutations: 7


### Manually adding a model to the list

In [12]:
# Optionally Adding in new models/tasks
def register_model(
    matrix_df: pd.DataFrame,
    ollama_name: str,
    task_overrides: dict | None = None,
    folder_name: str | None = None
) -> pd.DataFrame:
    """
    Register a new model into the evaluation matrix using only the Ollama name.

    Parameters
    ----------
    matrix_df : pd.DataFrame
        Existing evaluation matrix.
    ollama_name : str
        The Ollama model identifier, e.g., "llama3.1:70b".
    task_overrides : dict, optional
        Dict mapping task names -> status strings (e.g. {"sysengbench-c": "✅ samples: 2000"})
        Any task not included defaults to "❌".
    folder_name : str, optional
        Optional folder name; defaults to ollama_name with ':' replaced by '_'.

    Returns
    -------
    Updated matrix_df with the new model appended.
    """

    # Derive folder name if not supplied
    folder = folder_name or ollama_name.replace(":", "_")

    # Identify existing task columns
    task_columns = [
        c for c in matrix_df.columns
        if c not in ("Model Folder Name", "Model Ollama Name")
    ]

    # Initialize new row
    row = {
        "Model Folder Name": folder,
        "Model Ollama Name": ollama_name,
    }

    # Fill each task with overrides or default ❌
    for task in task_columns:
        if task_overrides and task in task_overrides:
            row[task] = task_overrides[task]
        else:
            row[task] = "❌"

    # Append row
    return pd.concat([matrix_df, pd.DataFrame([row])], ignore_index=True)



In [ ]:
# matrix_df = register_model(matrix_df, "gpt-oss:20b")
matrix_df = register_model(matrix_df, "gpt-oss:120b")
# IF I END UP HAVING TO RE-RUN MODELS, ADD THEM HERE:

# only flagship, mostly frontier models:
matrix_df = register_model(matrix_df, "deepseek-r1:70b")
matrix_df = register_model(matrix_df, "deepseek-r1:32b")
matrix_df = register_model(matrix_df, "gpt-oss:120b")  
matrix_df = register_model(matrix_df, "llama3.3:70b") 
matrix_df = register_model(matrix_df, "llama4:16x17b") 
matrix_df = register_model(matrix_df, "phi4:14b")               # new
matrix_df = register_model(matrix_df, "magistral:24b")     
matrix_df = register_model(matrix_df, "qwen3:32b")
matrix_df = register_model(matrix_df, "gemma3:27b")
matrix_df = register_model(matrix_df, "mistral-large:123b")

### add claude, chatgpt5, gemini

# smaller models or non-flagship
matrix_df = register_model(matrix_df, "deepseek-r1:8b")
matrix_df = register_model(matrix_df, "deepseek-r1:14b")
matrix_df = register_model(matrix_df, "gpt-oss:20b")                        

matrix_df = register_model(matrix_df, "phi4-mini:3.8b")         # new
matrix_df = register_model(matrix_df, "phi4-reasoning:14b")     
matrix_df = register_model(matrix_df, "qwen3:0.6b")             
matrix_df = register_model(matrix_df, "qwen3:1.7b")
matrix_df = register_model(matrix_df, "qwen3:4b")
matrix_df = register_model(matrix_df, "qwen3:8b")
matrix_df = register_model(matrix_df, "qwen3:14b")
matrix_df = register_model(matrix_df, "qwen3:30b")

matrix_df = register_model(matrix_df, "gemma3:1b")              
matrix_df = register_model(matrix_df, "gemma3:4b")
matrix_df = register_model(matrix_df, "gemma3:12b")
matrix_df = register_model(matrix_df, "gemma3:270m")
matrix_df = register_model(matrix_df, "gemma3n:e2b")            
matrix_df = register_model(matrix_df, "gemma3n:e4b")            
matrix_df = register_model(matrix_df, "llama3.2:1b")           
matrix_df = register_model(matrix_df, "llama3.2:3b") 

matrix_df = register_model(matrix_df, "mistral:7b")             
matrix_df = register_model(matrix_df, "mistral-small3.2:24b")   


In [14]:
display(matrix_df)

,Model Folder Name,Model Ollama Name,sysengbench,sysengbench-a,sysengbench-b,sysengbench-c,sysengbench-d,sysengbench-osq
0,deepseek-r1__14b,deepseek-r1:14b,✅ (1) – max:0.142 – samples:1144,✅ (2) – max:0.647 – samples:1144,✅ (2) – max:0.092 – samples:1144,✅ (2) – max:0.085 – samples:1144,✅ (2) – max:0.180 – samples:1144,❌ (1) – max:0.000 – samples:0
1,deepseek-r1__32b,deepseek-r1:32b,✅ (1) – max:0.145 – samples:1144,✅ (2) – max:0.656 – samples:1144,✅ (2) – max:0.087 – samples:1144,✅ (2) – max:0.079 – samples:1144,✅ (2) – max:0.176 – samples:1144,❌ (1) – max:0.000 – samples:0
2,deepseek-r1__8b,deepseek-r1:8b,✅ (1) – max:0.222 – samples:1144,✅ (2) – max:0.556 – samples:1144,✅ (2) – max:0.251 – samples:1144,✅ (2) – max:0.051 – samples:1144,❌ (1) – max:0.000 – samples:0,❌ (1) – max:0.000 – samples:0
3,devstral__24b,devstral:24b,✅ (1) – max:0.931 – samples:1144,✅ (1) – max:0.898 – samples:1144,✅ (1) – max:0.941 – samples:1144,✅ (1) – max:0.941 – samples:1144,✅ (1) – max:0.911 – samples:1144,✅ (1) – max:0.024 – samples:845
4,gemma3__12b,gemma3:12b,✅ (2) – max:0.892 – samples:1144,✅ (2) – max:0.865 – samples:1144,✅ (2) – max:0.894 – samples:1144,✅ (2) – max:0.906 – samples:1144,✅ (1) – max:0.892 – samples:1144,✅ (1) – max:0.015 – samples:845
5,gemma3__1b,gemma3:1b,✅ (2) – max:0.691 – samples:1144,✅ (2) – max:0.748 – samples:1144,✅ (2) – max:0.552 – samples:1144,✅ (2) – max:0.794 – samples:1144,✅ (1) – max:0.753 – samples:1144,❌ (1) – max:0.000 – samples:0
6,gemma3__270m,gemma3:270m,✅ (2) – max:0.136 – samples:1144,✅ (2) – max:0.108 – samples:1144,✅ (2) – max:0.007 – samples:1144,✅ (2) – max:0.317 – samples:1144,✅ (1) – max:0.010 – samples:1144,❌ (1) – max:0.000 – samples:0
7,gemma3__27b,gemma3:27b,✅ (2) – max:0.920 – samples:1144,✅ (2) – max:0.903 – samples:1144,✅ (2) – max:0.918 – samples:1144,✅ (2) – max:0.932 – samples:1144,✅ (1) – max:0.911 – samples:1144,❌ (1) – max:0.000 – samples:0
8,gemma3__4b,gemma3:4b,✅ (2) – max:0.863 – samples:1144,✅ (2) – max:0.828 – samples:1144,✅ (2) – max:0.844 – samples:1144,✅ (2) – max:0.876 – samples:1144,✅ (1) – max:0.847 – samples:1144,✅ (1) – max:0.007 – samples:845
9,gemma3n__e2b,gemma3n:e2b,✅ (2) – max:0.865 – samples:1144,✅ (2) – max:0.816 – samples:1144,✅ (2) – max:0.898 – samples:1144,✅ (2) – max:0.876 – samples:1144,✅ (1) – max:0.792 – samples:1144,❌ (1) – max:0.000 – samples:0


In [15]:
# assuming you already created the matrix with scores and samples
# matrix_df = generate_model_task_matrix_with_scores_and_samples(base_output_dir)

missing_dict = extract_missing_from_matrix(matrix_df)

print(missing_dict)

# Count all model-task pairs
total_permutations = sum(len(tasks) for tasks in missing_dict.values())

print(f"Total model-task permutations: {total_permutations}")

{'phi4-reasoning:14b': ['sysengbench-osq'], 'qwen3:0.6b': ['sysengbench-osq'], 'qwen3:14b': ['sysengbench-osq'], 'qwen3:32b': ['sysengbench-a', 'sysengbench-osq'], 'qwen3:4b': ['sysengbench-osq'], 'qwen3:8b': ['sysengbench-osq'], 'gpt-oss:120b': ['sysengbench', 'sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d', 'sysengbench-osq']}
Total model-task permutations: 13


# Running Multiple RunPod Containers

In [16]:
import os
import time
import runpod
import paramiko  # for SSH
from pathlib import Path

# ───────────────────────────────────────────────
# CONFIGURATION
# ───────────────────────────────────────────────
RUNPOD_API_KEY = os.environ["RUNPOD_API_KEY"]  # set in your environment
runpod.api_key = RUNPOD_API_KEY

## Read in the YAMLS

In [17]:
with open('sysengbench.yaml') as f:
    sysengbench_yaml = f.read()

with open('sysengbench-a.yaml') as f:
    sysengbench_a_yaml = f.read()

with open('sysengbench-b.yaml') as f:
    sysengbench_b_yaml = f.read()

with open('sysengbench-c.yaml') as f:
    sysengbench_c_yaml = f.read()

with open('sysengbench-d.yaml') as f:
    sysengbench_d_yaml = f.read()

with open('sysengbench-osq.yaml') as f:
    sysengbench_osq_yaml = f.read()

In [18]:
yaml_templates = {
    "sysengbench": sysengbench_yaml,
    "sysengbench-a": sysengbench_a_yaml,
    "sysengbench-b": sysengbench_b_yaml,
    "sysengbench-c": sysengbench_c_yaml,
    "sysengbench-d": sysengbench_d_yaml,
    "sysengbench-osq": sysengbench_osq_yaml,  # reuse base for osqa
}


## Sequential: Checks for each step to be done before proceeding on.

In [ ]:
import os, time, stat, posixpath, paramiko, runpod
from pathlib import Path

def run_missing_models(missing_dict, image_name, gpu_type,
                       yaml_templates, base_output_dir, local_results_dir):
    """
    Loop over {model: [tasks]} and run each missing task in its own RunPod pod.
    Includes blocking waits, exit-code checks, and robust downloading.
    """
    ssh_key_path = os.path.expanduser("~/.ssh/id_ed25519")

    def run_and_check(ssh, cmd, desc):
        """Run a remote command and wait for completion, printing output and errors."""
        print(f"▶ {desc}: {cmd}")
        stdin, stdout, stderr = ssh.exec_command(cmd)
        exit_code = stdout.channel.recv_exit_status()
        out = stdout.read().decode()
        err = stderr.read().decode()
        if out: print(out)
        if err: print("stderr:", err)
        if exit_code != 0:
            raise RuntimeError(f"❌ Command failed [{desc}] with exit {exit_code}")
        print(f"✔ {desc} finished.")
        return out

    def download_dir(sftp, remote_dir, local_dir):
        """Safely download a remote directory tree if it exists."""
        try:
            entries = sftp.listdir_attr(remote_dir)
        except FileNotFoundError:
            print(f"⚠ No output directory found at {remote_dir}")
            return
        os.makedirs(local_dir, exist_ok=True)
        for entry in entries:
            remote_path = posixpath.join(remote_dir, entry.filename)
            local_path  = os.path.join(local_dir, entry.filename)
            if stat.S_ISDIR(entry.st_mode):
                download_dir(sftp, remote_path, local_path)
            else:
                sftp.get(remote_path, local_path)
                print(f"  ↓ {local_path}")

    for model, tasks in missing_dict.items():
        for task in tasks:
            print(f"\n=== Starting pod for {model} | task: {task} ===")
            pod_name = f"lm-eval-{model.replace(':','-')}-{task}-{int(time.time())}"
            pod = runpod.create_pod(
                name=pod_name,
                image_name=image_name,
                gpu_type_id=gpu_type,
                gpu_count=1,
                container_disk_in_gb=200,
                min_vcpu_count=4,
                min_memory_in_gb=16,
                ports="22/tcp,11434/http",
                env={"OLLAMA_HOST": "0.0.0.0", "PYTHONUNBUFFERED": "1"},
                support_public_ip=True,
                start_ssh=True
            )
            pod_id = pod["id"]
            print(f"Created pod: {pod_id}")

            # Wait for pod to be RUNNING and runtime ports available
            ssh_host = ssh_port = None
            while True:
                details = runpod.get_pod(pod_id)
                status = details.get("desiredStatus")
                print(f"  Current status: {status}")
                if status == "RUNNING":
                    runtime = details.get("runtime")
                    if runtime and runtime.get("ports"):
                        for p in runtime["ports"]:
                            if p["type"]=="tcp" and p["privatePort"]==22 and p["isIpPublic"]:
                                ssh_host, ssh_port = p["ip"], p["publicPort"]
                                break
                        if ssh_host:
                            break
                time.sleep(10)

            print(f"Pod running at {ssh_host}:{ssh_port}")

            ssh = paramiko.SSHClient()
            ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
            ssh.connect(ssh_host, port=ssh_port, username="root",
                        pkey=paramiko.Ed25519Key.from_private_key_file(ssh_key_path))
            print("SSH connected.")

            try:
                # Provision environment
                provisioning_cmds = [
                    ("Install lshw", "apt update && apt install -y lshw"),
                    ("Install Ollama", "curl -fsSL https://ollama.com/install.sh | sh"),
                    ("Start Ollama", "nohup env OLLAMA_HOST=0.0.0.0 ollama serve > /tmp/ollama.log 2>&1 &"),
                    ("Wait for Ollama", "sleep 10"),
                    ("Install lm_eval + Ollama Python", "pip install lm_eval lm_eval[api] ollama==0.3.3"),
                ]
                for desc, cmd in provisioning_cmds:
                    run_and_check(ssh, cmd, desc)

                # Pull the model
                run_and_check(ssh, f"ollama pull {model}", f"Pull model {model}")

                # Push task YAML
                yaml_path = f"/root/{task}.yaml"
                yaml_content = yaml_templates[task]
                push_cmd = f"cat > {yaml_path} <<'EOF'\n{yaml_content}\nEOF"
                run_and_check(ssh, push_cmd, f"Upload YAML for {task}")

                # Run lm_eval and wait until it exits
                output_dir = f"output/{task}/"
                lm_eval_cmd = f"""
                lm_eval \
                  --model local-chat-completions \
                  --model_args model='{model}',base_url='http://localhost:11434/v1/chat/completions',num_concurrent=1 \
                  --include_path ./ \
                  --tasks {task} \
                  --output {output_dir} \
                  --log_samples \
                  --num_fewshot 0 \
                  --batch_size auto \
                  --gen_kwargs temperature=0.0 \
                  --apply_chat_template
                """
                run_and_check(ssh, lm_eval_cmd, f"Run lm_eval for {task}")

                # Download results safely
                sftp = ssh.open_sftp()
                remote_dir = f"/root/{output_dir}"
                local_dir = os.path.join(local_results_dir, task)
                download_dir(sftp, remote_dir, local_dir)
                sftp.close()

            except Exception as e:
                print(f"❌ Error during run for {model} | task: {task}: {e}")

            finally:
                runpod.terminate_pod(pod_id)
                ssh.close()
                print(f"Pod {pod_id} terminated.")


In [45]:
run_missing_models(
    missing_dict=missing_dict,
    image_name=IMAGE_NAME,
    gpu_type=GPU_TYPE,
    yaml_templates=yaml_templates,
    base_output_dir="output",
    local_results_dir=local_results_dir
)



=== Starting pod for mistral:7b | task: sysengbench-c ===
raw_response: {'data': {'podFindAndDeployOnDemand': {'id': 'petfrcbjud7jay', 'imageName': 'runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04', 'env': ['OLLAMA_HOST=0.0.0.0', 'PYTHONUNBUFFERED=1', 'PUBLIC_KEY=ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIB0U12zQAIppWu15PQqpNJHZd7+uFrYwuqP6CuH03FI2 ryan.a.bell2.civ@us.navy.mil\n'], 'machineId': 'j65nym2f1yl5', 'machine': {'podHostId': 'petfrcbjud7jay-64411291'}}}}
Created pod: petfrcbjud7jay
  Current status: RUNNING
  Current status: RUNNING
  Current status: RUNNING
  Current status: RUNNING
Pod running at 194.68.245.44:22160
SSH connected.
▶ Install lshw: apt update && apt install -y lshw
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1581 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2008 kB]
Get:4 http://

## Parallelizing

In [19]:
import os, time, stat, posixpath, paramiko, runpod
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

def run_single_model_task(model, task,
                          image_name, gpu_type,
                          yaml_templates,
                          base_output_dir,
                          local_results_dir,
                          log_dir):
    """
    Run exactly one model-task evaluation in its own RunPod pod.
    Returns (model, task, success_boolean).
    """
    log_file = Path(log_dir) / f"{model.replace(':','_')}__{task}.log"
    log_file.parent.mkdir(parents=True, exist_ok=True)

    def log(msg):
        stamp = time.strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{stamp}] [{model} | {task}] {msg}"
        print(line)
        with open(log_file, "a", encoding="utf-8") as lf:
            lf.write(line + "\n")

    def run_and_check(ssh, cmd, desc):
        log(f"▶ {desc}")
        stdin, stdout, stderr = ssh.exec_command(cmd)
        exit_code = stdout.channel.recv_exit_status()
        out = stdout.read().decode()
        err = stderr.read().decode()
        if out: log(out)
        if err: log(f"stderr: {err}")
        if exit_code != 0:
            raise RuntimeError(f"{desc} failed with exit code {exit_code}")
        log(f"✔ {desc} finished.")

    def download_dir(sftp, remote_dir, local_dir):
        try:
            entries = sftp.listdir_attr(remote_dir)
        except FileNotFoundError:
            log(f"⚠ No output directory found at {remote_dir}")
            return
        os.makedirs(local_dir, exist_ok=True)
        for entry in entries:
            remote_path = posixpath.join(remote_dir, entry.filename)
            local_path  = os.path.join(local_dir, entry.filename)
            if stat.S_ISDIR(entry.st_mode):
                download_dir(sftp, remote_path, local_path)
            else:
                sftp.get(remote_path, local_path)
                log(f"↓ {local_path}")

    ssh_key_path = os.path.expanduser("~/.ssh/id_ed25519")

    try:
        # 1. Create pod
        log("Creating pod...")
        pod = runpod.create_pod(
            name=f"lm-eval-{model.replace(':','-')}-{task}-{int(time.time())}",
            image_name=image_name,
            gpu_type_id=gpu_type,
            gpu_count=1,  ########### Change for bigger models! ##################
            container_disk_in_gb=200,
            min_vcpu_count=4,
            min_memory_in_gb=16,
            ports="22/tcp,11434/http",
            env={"OLLAMA_HOST": "0.0.0.0", "PYTHONUNBUFFERED": "1"},
            support_public_ip=True,
            start_ssh=True
        )
        pod_id = pod["id"]
        log(f"Created pod: {pod_id}")

        # 2. Wait for pod to be RUNNING and runtime ports ready
        ssh_host = ssh_port = None
        while True:
            details = runpod.get_pod(pod_id)
            status = details.get("desiredStatus")
            log(f"Current status: {status}")
            if status == "RUNNING":
                runtime = details.get("runtime")
                if runtime and runtime.get("ports"):
                    for p in runtime["ports"]:
                        if p["type"]=="tcp" and p["privatePort"]==22 and p["isIpPublic"]:
                            ssh_host, ssh_port = p["ip"], p["publicPort"]
                            break
                    if ssh_host:
                        break
            time.sleep(10)
        log(f"Pod running at {ssh_host}:{ssh_port}")

        # 3. Connect via SSH
        ssh = paramiko.SSHClient()
        ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        ssh.connect(ssh_host, port=ssh_port, username="root",
                    pkey=paramiko.Ed25519Key.from_private_key_file(ssh_key_path))
        log("SSH connected.")

        # 4. Provision environment
        steps = [
            ("Install lshw", "apt update && apt install -y lshw"),
            ("Install Ollama", "curl -fsSL https://ollama.com/install.sh | sh"),
            ("Start Ollama", "nohup env OLLAMA_HOST=0.0.0.0 ollama serve > /tmp/ollama.log 2>&1 &"),
            ("Wait for Ollama", "sleep 10"),
            ("Install lm_eval + Ollama Python", "pip install lm_eval lm_eval[api] ollama==0.3.3"),
            (f"Pull model {model}", f"ollama pull {model}")
        ]
        for desc, cmd in steps:
            run_and_check(ssh, cmd, desc)

        # 5. Upload task YAML
        yaml_path = f"/root/{task}.yaml"
        yaml_content = yaml_templates[task]
        run_and_check(ssh, f"cat > {yaml_path} <<'EOF'\n{yaml_content}\nEOF", f"Upload YAML for {task}")

        # 6. Run lm_eval and wait
        # optional no-thinking approach:
        # --model_args model='{model}',base_url='http://localhost:11434/v1/chat/completions',num_concurrent=1,think=false \
        output_dir = f"output/{task}/"
        lm_eval_cmd = f"""
        lm_eval \
          --model local-chat-completions \
          --model_args model='{model}',base_url='http://localhost:11434/v1/chat/completions',num_concurrent=1 \
          --include_path ./ \
          --tasks {task} \
          --output {output_dir} \
          --log_samples \
          --num_fewshot 0 \
          --batch_size auto \
          --gen_kwargs temperature=0.0 \
          --apply_chat_template
        """
        run_and_check(ssh, lm_eval_cmd, f"Run lm_eval for {task}")

        # 7. Download results
        sftp = ssh.open_sftp()
        remote_dir = f"/root/{output_dir}"
        local_dir  = os.path.join(local_results_dir, task)
        download_dir(sftp, remote_dir, local_dir)
        sftp.close()

        log("Job completed successfully.")
        return (model, task, True)

    except Exception as e:
        log(f"❌ Error: {e}")
        return (model, task, False)

    finally:
        try:
            runpod.terminate_pod(pod_id)
            log(f"Pod {pod_id} terminated.")
        except Exception as e:
            log(f"⚠ Pod termination failed: {e}")
        if 'ssh' in locals():
            ssh.close()


In [20]:
# Parallel Driver
def run_missing_models_parallel(missing_dict, image_name, gpu_type,
                                yaml_templates, base_output_dir,
                                local_results_dir, log_dir,
                                max_concurrent=3):
    """
    Run all missing model-task pairs with up to `max_concurrent` pods at once.
    Logs each job to its own file and prints live progress grouped by job.
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed
    Path(log_dir).mkdir(parents=True, exist_ok=True)

    futures = []
    with ThreadPoolExecutor(max_workers=max_concurrent) as executor:
        for model, tasks in missing_dict.items():
            for task in tasks:
                futures.append(
                    executor.submit(
                        run_single_model_task,
                        model, task,
                        image_name, gpu_type,
                        yaml_templates,
                        base_output_dir,
                        local_results_dir,
                        log_dir
                    )
                )

        # As each job finishes, print a concise summary
        for fut in as_completed(futures):
            model, task, success = fut.result()
            print(f"[SUMMARY] {model} | {task} → {'✅ success' if success else '❌ failed'}")


In [21]:
# missing_dict = {
#     "mistral:7b": ["sysengbench-c"],
#     "mistral:instruct": ["sysengbench-c"]
# }

IMAGE_NAME     = "runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04"
# GPU_TYPE       = "NVIDIA A40"
GPU_TYPE       = "NVIDIA H200"
local_results_dir = r"C:\Users\rabel\Desktop\dissertation-outputs\runpod_results"
log_dir        = r"C:\Users\rabel\Desktop\dissertation-outputs\runpod_logs"

run_missing_models_parallel(
    missing_dict,
    IMAGE_NAME,
    GPU_TYPE,
    yaml_templates,
    base_output_dir="output",
    local_results_dir=local_results_dir,
    log_dir=log_dir,
    max_concurrent=2   # run 4 pods at the same time
)


[2025-11-11 12:52:46] [phi4-reasoning:14b | sysengbench-osq] Creating pod...
[2025-11-11 12:52:46] [qwen3:0.6b | sysengbench-osq] Creating pod...
raw_response: {'data': {'podFindAndDeployOnDemand': {'id': '1fjbzvej46lttg', 'imageName': 'runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04', 'env': ['OLLAMA_HOST=0.0.0.0', 'PYTHONUNBUFFERED=1', 'PUBLIC_KEY=ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIB0U12zQAIppWu15PQqpNJHZd7+uFrYwuqP6CuH03FI2 ryan.a.bell2.civ@us.navy.mil\n'], 'machineId': 'mkfw0h5karf9', 'machine': {'podHostId': '1fjbzvej46lttg-64411516'}}}}
[2025-11-11 12:52:47] [qwen3:0.6b | sysengbench-osq] Created pod: 1fjbzvej46lttg
raw_response: {'data': {'podFindAndDeployOnDemand': {'id': 'l1yw4tbwne0d4k', 'imageName': 'runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04', 'env': ['OLLAMA_HOST=0.0.0.0', 'PYTHONUNBUFFERED=1', 'PUBLIC_KEY=ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIB0U12zQAIppWu15PQqpNJHZd7+uFrYwuqP6CuH03FI2 ryan.a.bell2.civ@us.navy.mil\n'], 'machineId': 'xp3

# Terminate ALL Pods. BE CAREFUL

## lm-eval pods only

In [4]:
import runpod

def terminate_lm_eval_pods():
    """
    Find every RunPod pod whose name starts with 'lm-eval-' and terminate it.
    """
    pods = runpod.get_pods()   # list all pods in your account
    if not pods:
        print("No pods found.")
        return

    # Filter only pods created by the evaluation runner
    lm_eval_pods = [p for p in pods if p.get("name", "").startswith("lm-eval-")]
    if not lm_eval_pods:
        print("No lm-eval pods to terminate.")
        return

    print(f"Found {len(lm_eval_pods)} lm-eval pods.")
    for pod in lm_eval_pods:
        pod_id = pod["id"]
        name   = pod["name"]
        status = pod.get("desiredStatus")
        print(f"Terminating {name} ({pod_id}) – current status: {status}")
        try:
            runpod.terminate_pod(pod_id)
            print(f"  ✔ Terminated {name} ({pod_id})")
        except Exception as e:
            print(f"  ❌ Could not terminate {name} ({pod_id}): {e}")

if __name__ == "__main__":
    terminate_lm_eval_pods()


Found 4 lm-eval pods.
Terminating lm-eval-qwen3-32b-sysengbench-a-1762875088 (759fp5xy764a08) – current status: RUNNING
  ✔ Terminated lm-eval-qwen3-32b-sysengbench-a-1762875088 (759fp5xy764a08)
Terminating lm-eval-phi4-reasoning-14b-sysengbench-osq-1762875088 (9hjj7i23z4iycp) – current status: RUNNING
  ✔ Terminated lm-eval-phi4-reasoning-14b-sysengbench-osq-1762875088 (9hjj7i23z4iycp)
Terminating lm-eval-qwen3-8b-sysengbench-osq-1762882892 (rh64jhe9pc79zj) – current status: RUNNING
  ✔ Terminated lm-eval-qwen3-8b-sysengbench-osq-1762882892 (rh64jhe9pc79zj)
Terminating lm-eval-qwen3-32b-sysengbench-osq-1762876489 (tmhqhpuwqfkeb2) – current status: RUNNING
  ✔ Terminated lm-eval-qwen3-32b-sysengbench-osq-1762876489 (tmhqhpuwqfkeb2)


# ALL PODS

In [ ]:
import runpod

def terminate_all_pods():
    """
    List all pods in your RunPod account and terminate every one.
    Prints each ID and status.
    """
    pods = runpod.get_pods()   # retrieves all pods you own
    if not pods:
        print("No pods found.")
        return

    print(f"Found {len(pods)} pods.")
    for pod in pods:
        pod_id = pod.get("id")
        name   = pod.get("name")
        status = pod.get("desiredStatus")
        print(f"Terminating {name} ({pod_id}) – current status: {status}")
        try:
            runpod.terminate_pod(pod_id)
            print(f"  ✔ Terminated {name} ({pod_id})")
        except Exception as e:
            print(f"  ❌ Could not terminate {name} ({pod_id}): {e}")

if __name__ == "__main__":
    terminate_all_pods()
